# Data Science Pipeline Project

This notebook builds a machine learning pipeline for StyleSense review recommendation prediction. The pipeline combines numerical, categorical, and NLP text features, then tunes and evaluates a classifier using held-out test data.


## 1. Load and Inspect Data

The target is `Recommended IND`. The features include numeric fields, categorical product metadata, and two text fields (`Title` and `Review Text`).


In [ ]:
import pandas as pd

df = pd.read_csv("data/reviews.csv")
print(df.shape)
print(df.dtypes)
print(df["Recommended IND"].value_counts(normalize=True))
df.head()


(18442, 9)
Clothing ID                 int64
Age                         int64
Title                      object
Review Text                object
Positive Feedback Count     int64
Division Name              object
Department Name            object
Class Name                 object
Recommended IND             int64

Recommended IND rate:\nRecommended IND
1    0.816235
0    0.183765\n

## 2. Prepare Features and Train/Test Split

The split is stratified so the train and test sets preserve the original recommendation class balance.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Recommended IND", axis=1)
y = df["Recommended IND"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=27, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])


Training rows: 14753\nTest rows: 3689\n

## 3. Build NLP and Structured Feature Transformers

The custom tokenizer normalizes words and applies a light suffix-based stemmer. The `TextStats` transformer creates extra NLP-derived features such as word count, punctuation counts, and simple positive/negative lexicon counts.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.column].fillna("")


class TextStats(BaseEstimator, TransformerMixin):
    negative_words = {"bad", "poor", "cheap", "disappoint", "return", "small", "large", "tight", "itchy"}
    positive_words = {"love", "great", "perfect", "comfortable", "beautiful", "recommend", "favorite"}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        text = pd.Series(X).fillna("").astype(str)
        words = text.str.lower().str.findall(r"[a-z']+")
        word_count = words.map(len)
        char_count = text.str.len()
        exclamation_count = text.str.count("!")
        question_count = text.str.count(r"\?")
        negative_count = words.map(lambda items: sum(word in self.negative_words for word in items))
        positive_count = words.map(lambda items: sum(word in self.positive_words for word in items))
        return np.vstack([word_count, char_count, exclamation_count, question_count, negative_count, positive_count]).T


def normalize_and_tokenize(text):
    tokens = []
    for raw in str(text).lower().split():
        token = "".join(ch for ch in raw if ch.isalpha())
        if len(token) <= 2 or token in ENGLISH_STOP_WORDS:
            continue
        for suffix in ("ing", "ed", "ly", "es", "s"):
            if token.endswith(suffix) and len(token) > len(suffix) + 3:
                token = token[: -len(suffix)]
                break
        tokens.append(token)
    return tokens


## 4. Create the Full Pipeline

Numerical, categorical, review text, and title text features are processed inside a single scikit-learn pipeline. Null handling is part of the pipeline so training and inference use the same preprocessing logic.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_pipeline():
    numeric_features = ["Age", "Positive Feedback Count"]
    categorical_features = ["Clothing ID", "Division Name", "Department Name", "Class Name"]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
    ])

    review_pipeline = Pipeline([
        ("selector", TextSelector("Review Text")),
        ("features", FeatureUnion([
            ("tfidf", TfidfVectorizer(tokenizer=normalize_and_tokenize, token_pattern=None, max_features=2500)),
            ("stats", TextStats()),
        ])),
    ])

    title_pipeline = Pipeline([
        ("selector", TextSelector("Title")),
        ("tfidf", TfidfVectorizer(tokenizer=normalize_and_tokenize, token_pattern=None, max_features=600)),
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("review_text", review_pipeline, ["Review Text"]),
        ("title_text", title_pipeline, ["Title"]),
    ])

    return Pipeline([
        ("preprocess", preprocessing),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear")),
    ])


## 5. Tune Hyperparameters

Grid search uses cross-validation on the training data only. The test set is held out until final evaluation.


In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline = build_pipeline()
param_grid = {
    "preprocess__review_text__features__tfidf__ngram_range": [(1, 1), (1, 2)],
    "classifier__C": [0.5, 1.0],
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)


Best parameters: {'classifier__C': 1.0, 'preprocess__review_text__features__tfidf__ngram_range': (1, 2)}\nBest CV F1: 0.9264\n

## 6. Evaluate on Held-Out Test Data

The final metrics use only the test set. Accuracy is useful, but precision and recall are especially important because the target is imbalanced toward recommended products.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score

best_model = grid.best_estimator_
predictions = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))
print(classification_report(y_test, predictions))


Accuracy: 0.8897\nPrecision: 0.9680\nRecall: 0.8944\n              precision    recall  f1-score   support

           0       0.65      0.87      0.74       678
           1       0.97      0.89      0.93      3011

    accuracy                           0.89      3689
   macro avg       0.81      0.88      0.84      3689
weighted avg       0.91      0.89      0.90      3689


## 7. Conclusion

The completed pipeline handles each data type in the model itself: numerical imputation/scaling, categorical imputation/one-hot encoding, and text tokenization/vectorization. The model is tuned with grid search and evaluated only after training using held-out data, meeting the project requirements for preprocessing, NLP feature engineering, fine-tuning, and model evaluation.
